In [1]:
# Cell 1: Imports & Load Data
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Paths
DATA_DIR = r"C:\Users\arnaw\OneDrive\Desktop\transfer_recomender\data"
MODELS_DIR = r"C:\Users\arnaw\OneDrive\Desktop\transfer_recomender\models"

# Load clustered data
fw = pd.read_csv(os.path.join(DATA_DIR, "fw_clustered.csv"))
mf = pd.read_csv(os.path.join(DATA_DIR, "mf_clustered.csv"))
df = pd.read_csv(os.path.join(DATA_DIR, "df_clustered.csv"))

# Load original data for team info
players = pd.read_csv(os.path.join(DATA_DIR, "players_data-2024_2025.csv"))

print(f"FW: {fw.shape}")
print(f"MF: {mf.shape}")
print(f"DF: {df.shape}")
print(f"All players: {players.shape}")
print(f"\nTeams in dataset: {players['Squad'].nunique()}")
print(f"\nSample teams:")
print(players['Squad'].unique()[:10])

FW: (720, 223)
MF: (900, 223)
DF: (1022, 223)
All players: (2854, 267)

Teams in dataset: 96

Sample teams:
['Bournemouth' 'Valencia' 'Udinese' 'Marseille' 'Saint-Étienne' 'Angers'
 'Nice' 'Roma' 'Osasuna' 'Getafe']


In [2]:
# Cell 2: Build Team Profile
def get_team_profile(team_name):
    
    # check if team exists
    team_players = players[players['Squad'] == team_name]
    
    if team_players.empty:
        print(f"Team '{team_name}' not found!")
        print("Available teams:", players['Squad'].unique())
        return None
    
    print(f"Team: {team_name}")
    print(f"Total players found: {len(team_players)}")
    print(f"Positions: {team_players['Pos'].unique()}")
    
    return team_players

# Test with Liverpool
team = get_team_profile('Liverpool')
print(team[['Player', 'Pos', 'Age']].head(10))

Team: Liverpool
Total players found: 24
Positions: ['DF' 'GK' 'FW' 'MF' 'MF,DF']
                      Player    Pos   Age
83    Trent Alexander-Arnold     DF  25.0
87                   Alisson     GK  31.0
378            Conor Bradley     DF  21.0
547          Federico Chiesa     FW  26.0
658             Jayden Danns     MF  18.0
723                Luis Díaz     FW  27.0
832           Harvey Elliott     MF  21.0
844              Wataru Endo  MF,DF  31.0
955               Cody Gakpo     FW  25.0
1028               Joe Gomez     DF  27.0


In [6]:
# Cell 3: Detect Team Weakness
def detect_weakness(team_name, position):
    
    # get position data
    if position == 'FW':
        pos_data = fw
        pos_filter = ['FW', 'FW,MF', 'MF,FW']
    elif position == 'MF':
        pos_data = mf
        pos_filter = ['MF', 'MF,DF', 'DF,MF', 'MF,FW', 'FW,MF']
    else:
        pos_data = df
        pos_filter = ['DF', 'DF,MF', 'MF,DF']
    
    # get numeric cols
    numeric_cols = pos_data.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c != 'cluster']
    
    # get league average
    league_avg = pos_data[numeric_cols].mean()
    
    # get team players in that position
    team_players = players[
        (players['Squad'] == team_name) & 
        (players['Pos'].isin(pos_filter))
    ]
    
    if team_players.empty:
        print(f"No {position} players found for {team_name}")
        return None
    
    # get team's position players from clustered data
    team_pos_players = pos_data[
        pos_data['Player'].isin(team_players['Player'])
    ]
    
    if team_pos_players.empty:
        print(f"No matching players found in clustered data")
        return None
    
    # get team average for that position
    team_avg = team_pos_players[numeric_cols].mean()
    
    # compare team vs league
    weakness_score = (league_avg - team_avg).mean()
    
    print(f"\n{team_name} — {position} Analysis")
    print(f"Players found: {len(team_pos_players)}")
    print(f"Weakness score: {weakness_score:.4f}")
    print(f"(Positive = below league average = weakness)")
    
    return weakness_score, team_avg, league_avg

# Test
detect_weakness('Liverpool', 'FW')
detect_weakness('Liverpool', 'MF')
detect_weakness('Liverpool', 'DF')


Liverpool — FW Analysis
Players found: 6
Weakness score: -55.6192
(Positive = below league average = weakness)

Liverpool — MF Analysis
Players found: 7
Weakness score: -50.4891
(Positive = below league average = weakness)

Liverpool — DF Analysis
Players found: 8
Weakness score: -119.4901
(Positive = below league average = weakness)


(np.float64(-119.4900928263212),
 Rk          1570.625
 Age           26.250
 Born        1997.625
 MP            24.125
 Starts        18.750
               ...   
 PSxG+/-          NaN
 /90              NaN
 Att (GK)         NaN
 #OPA             NaN
 #OPA/90          NaN
 Length: 192, dtype: float64,
 Rk          1426.801370
 Age           25.243137
 Born        1998.414706
 MP            18.799413
 Starts        14.907045
                ...     
 PSxG+/-             NaN
 /90                 NaN
 Att (GK)            NaN
 #OPA                NaN
 #OPA/90             NaN
 Length: 192, dtype: float64)

In [7]:
# Cell 4: Find Weakest Position
def find_weakest_position(team_name):
    
    scores = {}
    
    fw_result = detect_weakness(team_name, 'FW')
    mf_result = detect_weakness(team_name, 'MF')
    df_result = detect_weakness(team_name, 'DF')
    
    if fw_result: scores['FW'] = fw_result[0]
    if mf_result: scores['MF'] = mf_result[0]
    if df_result: scores['DF'] = df_result[0]
    
    # weakest = highest score (least above average)
    weakest = max(scores, key=scores.get)
    
    print(f"\n{'='*40}")
    print(f"{team_name} Position Scores:")
    for pos, score in scores.items():
        print(f"  {pos}: {score:.2f}")
    print(f"\nWeakest position: {weakest} ← recommend here")
    print(f"{'='*40}")
    
    return weakest

# Test
find_weakest_position('Liverpool')


Liverpool — FW Analysis
Players found: 6
Weakness score: -55.6192
(Positive = below league average = weakness)

Liverpool — MF Analysis
Players found: 7
Weakness score: -50.4891
(Positive = below league average = weakness)

Liverpool — DF Analysis
Players found: 8
Weakness score: -119.4901
(Positive = below league average = weakness)

Liverpool Position Scores:
  FW: -55.62
  MF: -50.49
  DF: -119.49

Weakest position: MF ← recommend here


'MF'

In [14]:
# Cell 5: Get Recommendations
def get_recommendations(team_name, position=None, top_n=5):
    
    # auto detect weakness if position not given
    if position is None:
        print(f"Auto-detecting weakness for {team_name}...")
        position = find_weakest_position(team_name)
    
    print(f"\nFinding {position} recommendations for {team_name}...")
    
    # get position data
    if position == 'FW':
        pos_data = fw
        pos_filter = ['FW', 'FW,MF', 'MF,FW']
    elif position == 'MF':
        pos_data = mf
        pos_filter = ['MF', 'MF,DF', 'DF,MF', 'MF,FW', 'FW,MF']
    else:
        pos_data = df
        pos_filter = ['DF', 'DF,MF', 'MF,DF']
    
    # get numeric cols
    numeric_cols = pos_data.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c != 'cluster']
    
    # get team's players in that position
    team_players = players[
        (players['Squad'] == team_name) &
        (players['Pos'].isin(pos_filter))
    ]
    
    # get team players from clustered data
    team_pos_players = pos_data[
        pos_data['Player'].isin(team_players['Player'])
    ]
    
    if team_pos_players.empty:
        print(f"No {position} players found for {team_name}")
        return None
    
    # build ideal profile
    team_avg = team_pos_players[numeric_cols].mean().fillna(0)
    
    # get all players NOT from this team
    other_players = pos_data[
        ~pos_data['Player'].isin(team_players['Player'])
    ].copy()
    
    # calculate similarity
    team_vector = team_avg.values.reshape(1, -1)
    other_stats = other_players[numeric_cols].fillna(0).values
    
    scores = cosine_similarity(team_vector, other_stats)[0]
    other_players['similarity'] = scores
    
    # sort and get top N
    results = other_players.sort_values(
        'similarity', ascending=False
    ).head(top_n)
    
    # display results
    print(f"\n{'='*50}")
    print(f"Top {top_n} {position} recommendations for {team_name}")
    print(f"{'='*50}")
    print(results[['Player', 'Squad', 'Comp', 'similarity']].to_string(index=False))
    
    return results

# Test
get_recommendations('Liverpool', 'MF')


Finding MF recommendations for Liverpool...

Top 5 MF recommendations for Liverpool
          Player       Squad               Comp  similarity
      Arda Güler Real Madrid         es La Liga    0.998399
Teun Koopmeiners    Juventus         it Serie A    0.998060
 Boubacar Kamara Aston Villa eng Premier League    0.997852
Jack Hinshelwood    Brighton eng Premier League    0.997802
            Gavi   Barcelona         es La Liga    0.997416


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,CS,PSxG,PSxG+/-,/90,Att (GK),#OPA,#OPA/90,Primary_Pos,cluster,similarity
370,1097,Arda Güler,tr TUR,"MF,FW",Real Madrid,es La Liga,19.0,2005.0,28,14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MF,5,0.998399
462,1393,Teun Koopmeiners,nl NED,"MF,FW",Juventus,it Serie A,26.0,1998.0,28,23,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MF,5,0.998060
431,1310,Boubacar Kamara,fr FRA,MF,Aston Villa,eng Premier League,24.0,1999.0,26,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MF,5,0.997852
388,1166,Jack Hinshelwood,eng ENG,"MF,DF",Brighton,eng Premier League,19.0,2005.0,26,22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MF,5,0.997802
319,982,Gavi,es ESP,"MF,FW",Barcelona,es La Liga,19.0,2004.0,26,14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MF,5,0.997416


In [15]:
test_teams = ['Manchester City', 'Arsenal', 'Barcelona', 'Bayern Munich']

for team in test_teams:
    print(f"\n{'='*50}")
    get_recommendations(team)
    print()


Auto-detecting weakness for Manchester City...

Manchester City — FW Analysis
Players found: 9
Weakness score: -10.4353
(Positive = below league average = weakness)

Manchester City — MF Analysis
Players found: 8
Weakness score: -86.3902
(Positive = below league average = weakness)

Manchester City — DF Analysis
Players found: 14
Weakness score: -31.2213
(Positive = below league average = weakness)

Manchester City Position Scores:
  FW: -10.44
  MF: -86.39
  DF: -31.22

Weakest position: FW ← recommend here

Finding FW recommendations for Manchester City...

Top 5 FW recommendations for Manchester City
                Player       Squad               Comp  similarity
Matias Fernandez-Pardo       Lille         fr Ligue 1    0.992203
            Elif Elmas      Torino         it Serie A    0.987597
            Diogo Jota   Liverpool eng Premier League    0.986951
    Georges Mikautadze        Lyon         fr Ligue 1    0.986946
          Wahbi Khazri Montpellier         fr Ligue 1    0

In [ ]:
import joblib
import os

# save all functions are ready
print("✅ Recommender system ready!")
print("✅ get_recommendations() function working!")
print("✅ find_weakest_position() function working!")
print("✅ detect_weakness() function working!")

# save team list for app
team_list = sorted(players['Squad'].unique().tolist())
joblib.dump(team_list, os.path.join(MODELS_DIR, "team_list.pkl"))
print(f"\n✅ Team list saved! ({len(team_list)} teams)")
print("\nSample teams:")
print(team_list[:10])

✅ Recommender system ready!
✅ get_recommendations() function working!
✅ find_weakest_position() function working!
✅ detect_weakness() function working!

✅ Team list saved! (96 teams)

Sample teams:
['Alavés', 'Angers', 'Arsenal', 'Aston Villa', 'Atalanta', 'Athletic Club', 'Atlético Madrid', 'Augsburg', 'Auxerre', 'Barcelona']
